In [1]:
# import required library
import pandas as pd
from sqlalchemy import create_engine, inspect
import os
from dotenv import load_dotenv
load_dotenv()
from datetime import datetime
import json
import re

In [2]:
# Membuat koneksi database
def db_connection():
	src_database = os.getenv("SRC_POSTGRES_DB")
	src_host = os.getenv("SRC_POSTGRES_HOST")
	src_user = os.getenv("SRC_POSTGRES_USER")
	src_password = os.getenv("SRC_POSTGRES_PASSWORD")
	src_port = os.getenv("SRC_POSTGRES_PORT")
			
	src_conn = f'postgresql://{src_user}:{src_password}@{src_host}:{src_port}/{src_database}'

	src_engine = create_engine(src_conn)

	return src_engine

In [3]:
# 1. membuat fungsi unuk mengambil tabel di database sql
def get_table(engine):
    # Gunakan inspector untuk melihat tabel
	inspector = inspect(engine)
	table_names = inspector.get_table_names()

	print("Tabel yang ditemukan:", table_names)
	return table_names

In [4]:
# Tampilkan seluruh tabel
src_engine = db_connection()
list_table = get_table(src_engine)

Tabel yang ditemukan: ['products', 'inventory_tracking', 'orders', 'order_details', 'customers', 'employees']


In [5]:
# 2. extract data dalam database ke dataframe
def extract_data(list_table, src_engine):
    dict_data = {}
	
    for table_name in list_table:
        # Membuat query untuk extract database
        query = f"SELECT * FROM {table_name}"

		# Read data into DataFrame
        df = pd.read_sql_query(query, src_engine)
		
		# masukkan database ke dalam query
        dict_data[table_name] = df

    return dict_data

In [6]:
dict_extract_data = extract_data(list_table, src_engine)
dict_extract_data

{'products':     product_id            product_name  category unit_price cost_price  \
 0           53        Pastry situation    Pastry     $47.00     $43.00   
 1           54             Pastry sure    Pastry     $47.00     $43.00   
 2           55        Pastry statement    Pastry     $17.00     $14.00   
 3           56                Salad on     Salad     $21.00     $20.00   
 4           57            Salad really     Salad     $43.00     $40.00   
 5           58               Salad bag     Salad      $9.00      $1.00   
 6           59         Pastry security    Pastry      $2.00     $-3.00   
 7           60           Smoothie lead  Smoothie     $48.00     $45.00   
 8           61            Salad people     Salad     $10.00      $6.00   
 9           62           Sandwich role  Sandwich     $13.00      $9.00   
 10          63             Pastry plan    Pastry     $10.00      $2.00   
 11          64               Salad bag     Salad     $37.00     $34.00   
 12          

In [7]:
# 3. Ambil total row dan col pada dataframe
def get_rows_cols(dict_extract_data):
    dict_shape = {}

    for key, value in dict_extract_data.items():
        (n_rows, n_cols) = value.shape

        dict_shape[key] = [n_rows, n_cols]

    return dict_shape

In [8]:
dict_shape = get_rows_cols(dict_extract_data)
dict_shape

{'products': [54, 7],
 'inventory_tracking': [162, 6],
 'orders': [1010, 8],
 'order_details': [3022, 7],
 'customers': [204, 7],
 'employees': [103, 7]}

In [9]:
# 4. cek datatype semua kolom
def check_data_type(dict_extract_data):
    dict_data_type = {}

    for key, value in dict_extract_data.items():
        dict_data_type[key] = {}
        for column in value.columns:
            column_dtype = str(value[column].dtype)
            dict_data_type[key][column] = column_dtype

    return dict_data_type

In [10]:
dict_data_type = check_data_type(dict_extract_data)
dict_data_type

{'products': {'product_id': 'int64',
  'product_name': 'str',
  'category': 'str',
  'unit_price': 'str',
  'cost_price': 'str',
  'in_stock': 'bool',
  'created_at': 'datetime64[us]'},
 'inventory_tracking': {'tracking_id': 'int64',
  'product_id': 'int64',
  'quantity_change': 'int64',
  'change_date': 'datetime64[us]',
  'reason': 'str',
  'created_at': 'datetime64[us]'},
 'orders': {'order_id': 'int64',
  'employee_id': 'int64',
  'customer_id': 'float64',
  'order_date': 'datetime64[us]',
  'total_amount': 'float64',
  'payment_method': 'str',
  'order_status': 'str',
  'created_at': 'datetime64[us]'},
 'order_details': {'order_detail_id': 'int64',
  'order_id': 'int64',
  'product_id': 'int64',
  'quantity': 'int64',
  'unit_price': 'float64',
  'subtotal': 'float64',
  'created_at': 'datetime64[us]'},
 'customers': {'customer_id': 'int64',
  'first_name': 'str',
  'last_name': 'str',
  'email': 'str',
  'phone': 'str',
  'loyalty_points': 'int64',
  'created_at': 'datetime64[us]

In [11]:
# 5. Check unique value spesifik dataframe dan column
def check_unique_value(dict_extract_data):
    dict_unique_value = {}

    table_to_check = {
        'products': 'category',
        'inventory_tracking': 'reason',
        'orders': 'payment_method',
        'employees': 'role'
    }

    for key, value in table_to_check.items():
        list_unique_value = dict_extract_data[key][value].unique().tolist()

        dict_unique_value[value] = list_unique_value

    return dict_unique_value

In [12]:
dict_unique_value = check_unique_value(dict_extract_data)
dict_unique_value

{'category': ['Pastry', 'Salad', 'Smoothie', 'Sandwich', 'Coffee', 'Tea'],
 'reason': ['Restock', 'Damaged', 'Expired', 'ERROR'],
 'payment_method': ['Cash', 'Credit Card', 'Debit Card', 'ERROR'],
 'role': ['Cashier',
  'Waitress',
  'Manager',
  'Waiter',
  'Barista',
  'today',
  'third',
  'me']}

In [13]:
# 6. Menyimpan dat profiling ke dalam file JSON
def save_data(dict_extract_data, dict_shape, dict_data_type, dict_unique_value):
    data_profiling = {
        "person_in_charge": "Ilham",
        "date_profiling": f"{datetime.now()}",
        "result": {}
    }

    for key, value in dict_extract_data.items():
        data_profiling["result"][key] = {}
        data_profiling["result"][key]["shape"] = dict_shape[key]
        data_profiling["result"][key]["data_type"] = dict_data_type[key]
        data_profiling["result"][key]["unique_values"] = {}

        for column in value.columns:
            if column in dict_unique_value:
                data_profiling["result"][key]["unique_values"][column] = dict_unique_value[column]

    # Convert and write JSON object to file
    with open("data_profiling.json", "w") as f:
        f.write(json.dumps(data_profiling, indent = 4))

In [14]:
save_data(dict_extract_data, dict_shape, dict_data_type, dict_unique_value)

# Data Quality

In [15]:
# 1. check missing value
def check_mising_value(dict_extract_data):
    dict_missing_value = {}

    for key, value in dict_extract_data.items():
        dict_missing_value[key] = {}
        n_data = len(dict_extract_data[key])

        for column in value.columns:
            n_missing_column = value[column].isnull().sum()
            percentage_missing = round(n_missing_column*100/n_data, 2)

            dict_missing_value[key][column] = str(percentage_missing)

    return dict_missing_value

In [16]:
dict_missing_value = check_mising_value(dict_extract_data)
dict_missing_value

{'products': {'product_id': '0.0',
  'product_name': '0.0',
  'category': '0.0',
  'unit_price': '0.0',
  'cost_price': '0.0',
  'in_stock': '0.0',
  'created_at': '0.0'},
 'inventory_tracking': {'tracking_id': '0.0',
  'product_id': '0.0',
  'quantity_change': '0.0',
  'change_date': '0.0',
  'reason': '0.0',
  'created_at': '0.0'},
 'orders': {'order_id': '0.0',
  'employee_id': '0.0',
  'customer_id': '24.75',
  'order_date': '0.0',
  'total_amount': '0.0',
  'payment_method': '0.0',
  'order_status': '0.0',
  'created_at': '0.0'},
 'order_details': {'order_detail_id': '0.0',
  'order_id': '0.0',
  'product_id': '0.0',
  'quantity': '0.0',
  'unit_price': '0.0',
  'subtotal': '0.0',
  'created_at': '0.0'},
 'customers': {'customer_id': '0.0',
  'first_name': '0.0',
  'last_name': '0.0',
  'email': '0.0',
  'phone': '1.96',
  'loyalty_points': '0.0',
  'created_at': '0.0'},
 'employees': {'employee_id': '0.0',
  'first_name': '0.0',
  'last_name': '0.0',
  'hire_date': '0.0',
  'role

In [17]:
for key, value in dict_missing_value.items():
    print(f"missing value pada tabel {key}")
    print(value)

missing value pada tabel products
{'product_id': '0.0', 'product_name': '0.0', 'category': '0.0', 'unit_price': '0.0', 'cost_price': '0.0', 'in_stock': '0.0', 'created_at': '0.0'}
missing value pada tabel inventory_tracking
{'tracking_id': '0.0', 'product_id': '0.0', 'quantity_change': '0.0', 'change_date': '0.0', 'reason': '0.0', 'created_at': '0.0'}
missing value pada tabel orders
{'order_id': '0.0', 'employee_id': '0.0', 'customer_id': '24.75', 'order_date': '0.0', 'total_amount': '0.0', 'payment_method': '0.0', 'order_status': '0.0', 'created_at': '0.0'}
missing value pada tabel order_details
{'order_detail_id': '0.0', 'order_id': '0.0', 'product_id': '0.0', 'quantity': '0.0', 'unit_price': '0.0', 'subtotal': '0.0', 'created_at': '0.0'}
missing value pada tabel customers
{'customer_id': '0.0', 'first_name': '0.0', 'last_name': '0.0', 'email': '0.0', 'phone': '1.96', 'loyalty_points': '0.0', 'created_at': '0.0'}
missing value pada tabel employees
{'employee_id': '0.0', 'first_name':

In [18]:
# 2. Validasi format date
pattern = r'\b\d{4}\b-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])\b'
pattern_datetime = r'\b\d{4}\b-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01]) ([01][0-9]|2[0-4]):([0-5][0-9]|6[0]):([0-5][0-9]|6[0])\b'

def check_date_validate(string, pattern):
    # match the pattern using regex
    if re.match(pattern, string):
        return True
    
    else:
        return False


In [22]:
# buat kolom baru is_date_valid untuk beberapa tabel
dict_extract_data['employees']['is_valid_hire_date'] = dict_extract_data['employees']['hire_date'].astype(str).apply(check_date_validate, pattern=pattern)
dict_extract_data['inventory_tracking']['is_valid_change_date'] = dict_extract_data['inventory_tracking']['change_date'].astype(str).apply(check_date_validate, pattern=pattern)
dict_extract_data['orders']['is_valid_order_date'] = dict_extract_data['orders']['order_date'].astype(str).apply(check_date_validate, pattern=pattern_datetime)

In [23]:
# buat fungsi percentage date valid
def get_percentage_valid_data(dict_extract_data):
    # instansiasi dict kosong
    dictionary = {}

    # ambil tabel yang akan divalidasi
    table_to_check = {
        'employees': 'is_valid_hire_date',
        'inventory_tracking': 'is_valid_change_date',
        'orders': 'is_valid_order_date'
    }

    for key, df in dict_extract_data.items():
        if key in table_to_check:
            dictionary[key] = {}

            # filter data only get valid
            valid_data = df[df[table_to_check[key]] == True]

            # get length of data
            n_valid_data = len(valid_data)
            n_data = len(df)

            # calculate percentage
            percentage_valid = round((n_valid_data / n_data), 2)

            dictionary[key][table_to_check[key]] = percentage_valid

    return dictionary

In [24]:
# dapatkan nilai percentage
dict_percetage_valid_date = get_percentage_valid_data(dict_extract_data)
dict_percetage_valid_date

{'inventory_tracking': {'is_valid_change_date': 1.0},
 'orders': {'is_valid_order_date': 1.0},
 'employees': {'is_valid_hire_date': 1.0}}

In [25]:
# 3. validity numeric value
def check_numeric_type(dict_extract_data):
    dict_numeric = {}

    table_to_check = {
        'products': ['unit_price', 'cost_price'],
        'orders': ['total_amount'],
        'order_details': ['unit_price', 'quantity', 'subtotal'],
        'inventory_tracking': ['quantity_change'],
        'customers': ['loyalty_points']
    }
    
    for key, df in dict_extract_data.items():
        if key in table_to_check:
            dict_numeric[key] = {}

            for _, col_name in enumerate(table_to_check[key]):
                is_numeric = pd.api.types.is_numeric_dtype(df[col_name])
                dict_numeric[key][col_name] = is_numeric

    
    return dict_numeric

In [26]:
dict_numeric_type = check_numeric_type(dict_data_type)
dict_numeric_type

{'products': {'unit_price': False, 'cost_price': False},
 'inventory_tracking': {'quantity_change': True},
 'orders': {'total_amount': True},
 'order_details': {'unit_price': True, 'quantity': True, 'subtotal': True},
 'customers': {'loyalty_points': True}}

In [41]:
# 4. validasi negatif value
def check_negative_value(dict_extract_data):
    dict_negative = {}

    table_to_check = {
        'orders': ['total_amount'],
        'products': ['unit_price', 'cost_price'],
        'order_details': ['unit_price', 'quantity', 'subtotal'],
        'inventory_tracking': ['quantity_change'],
        'customers': ['loyalty_points']
    }
    
    for key, df in dict_extract_data.items():
        if key in table_to_check:
            dict_negative[key] = {}

            for _, col_name in enumerate(table_to_check[key]):
                if df[col_name].dtype == 'str' or df[col_name].dtype == 'object':
                    df[f'{col_name}_float'] = df[col_name].str.extract('(\d+)').astype(float)
                    is_negative = (df[f'{col_name}_float'] < 0).any()

                else:
                    is_negative = (df[col_name] < 0).any()
                    
                dict_negative[key][col_name] = str(is_negative)

    return dict_negative   

<>:19: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\d'
C:\Users\User\AppData\Local\Temp\ipykernel_2116\3720725849.py:19: SyntaxWarning: invalid escape sequence '\d'
  df[f'{col_name}_float'] = df[col_name].str.extract('(\d+)').astype(float)


In [42]:
dict_negative_value = check_negative_value(dict_extract_data)
dict_negative_value

{'products': {'unit_price': 'False', 'cost_price': 'False'},
 'inventory_tracking': {'quantity_change': 'False'},
 'orders': {'total_amount': 'False'},
 'order_details': {'unit_price': 'False',
  'quantity': 'False',
  'subtotal': 'False'},
 'customers': {'loyalty_points': 'True'}}

In [43]:
# 5. Membuat Report Data Quality
def save_data_quality(dict_extract_data, dict_missing_value, dict_percetage_valid_date, dict_numeric_type, dict_negative_value):
    data_quality = {
        "person_in_charge": "Ilham",
        "date_profiling": f"{datetime.now()}",
        "result": {}
    }

    for table in dict_extract_data.keys():
        data_quality["result"][table] = {}

        # Missing Values
        if table in dict_missing_value:
            data_quality["result"][table]["missing_values"] = dict_missing_value[table]

        # Date Validity
        if table in dict_percetage_valid_date:
            data_quality["result"][table]["date_validate"] = dict_percetage_valid_date[table]

        # Numeric Type
        if table in dict_numeric_type:
            data_quality["result"][table]["numeric_type"] = dict_numeric_type[table]

        # Negative Value
        if table in dict_negative_value:
            data_quality["result"][table]["negative_value"] = dict_negative_value[table]
            
    # Convert and write JSON object to file
    with open("data_quality.json", "w") as f:
        f.write(json.dumps(data_quality, indent = 4))   

In [44]:
save_data_quality(dict_extract_data, dict_missing_value, dict_percetage_valid_date, dict_numeric_type, dict_negative_value)

# Rekomendasi

1. Apabila ingin melakukan analisis customer dengan order terbanyak atau analisis lain yang berhubungan antara `orders` dan `customers`. Kita perlu lakukan handling pada missing value customer_id pada tabel orders karena jumlah yang hilang > 20%.

2. Dibuat constraint not null untuk column `customer_id`, `order_id`, dan column yang potensial menjadi primary key database.

3. Nilai pada kolom `unit_price` dan `cost_price` belum bertipe numeric. Terdapat tanda `$` di dalam nilainya yang dapat mengganggu proses aggresasi (seperti menjumlahkan, cari rata-rata dan semisalnya). Kita pelru hilangkan tada `$` tersebut dan memindahkan nilai mata uang ke dalam kolomnya (contoh `unit_price_usd`).

4. Kolom `phone` pada tabel customers dibiarkan saja karena hanya tabel opsional dan tidak mempengaruhi analisis secara langsung.

5. Kolom `loyalty_points` pada tabel customers memiliki nilai negatif. Untuk atisipasi, perlu adakan constraint nilai positif (>0) pada postgresql pada kolom ini dan beberapa kolom lainnya.